# GPT 2.0 — 한국어 문학으로 학습 (Colab)

Karpathy의 *"Let's build GPT"* (nn-zero-to-hero, Lecture 7)와 **동일한 구조**, 데이터만 tiny-Shakespeare 대신 **공개 한국어 근대문학**(김유정·현진건·김동인·이상·나도향·이효석·최서해·강경애 등, 한국어 위키문헌)으로 교체.

**사용법:** `런타임 → 런타임 유형 변경 → GPU(T4)` 선택 후 셀을 위에서부터 실행(Shift+Enter). 업로드 불필요.

## 1. GPU 확인

In [ ]:
import torch
device='cuda' if torch.cuda.is_available() else ('mps' if getattr(torch.backends,'mps',None) and torch.backends.mps.is_available() else 'cpu')
print('torch',torch.__version__,'| device =',device)
if device=='cpu': print('GPU 미감지: 런타임 유형을 GPU(T4)로 바꾸세요.')

## 2. 데이터셋 준비 (노트북에 내장 — 업로드 불필요)

In [ ]:
import gzip, base64
DATA_B64 = "..."
text = gzip.decompress(base64.b64decode(DATA_B64)).decode('utf-8')
open('input.txt','w',encoding='utf-8').write(text)
chars=sorted(set(text)); vocab_size=len(chars)
print(f'총 {len(text):,}자 | 어휘(고유문자) {vocab_size}개')
print(text[:200])

## 3. 토크나이저 + 데이터 분할 (문자 단위)

In [ ]:
stoi={c:i for i,c in enumerate(chars)}; itos={i:c for i,c in enumerate(chars)}
encode=lambda s:[stoi[c] for c in s]; decode=lambda l:''.join(itos[i] for i in l)
data=torch.tensor(encode(text),dtype=torch.long)
n=int(0.9*len(data)); train_data,val_data=data[:n],data[n:]
print('train',len(train_data),'| val',len(val_data))

## 4. 하이퍼파라미터

> 데이터가 약 52만 자라 강의(110만)보다 작습니다. 과적합을 줄이려고 `dropout`을 0.25로 약간 높였어요. 학습 중 **train은 내려가는데 val이 다시 올라가면 과적합** 신호이니, val이 최저였던 부근이 최선의 모델입니다.

In [ ]:
batch_size=64
block_size=256
max_iters=5000
eval_interval=250
eval_iters=200
learning_rate=3e-4
n_embd=384
n_head=6
n_layer=6
dropout=0.25
torch.manual_seed(1337)

## 5. 모델 정의 (강의와 동일한 Transformer)

In [ ]:
import torch.nn.functional as F
model.load_state_dict(torch.load('ckpt_best.pt')['model']); model.eval()   # val 최저(=과적합 전) 모델

@torch.no_grad()
def sample(prompt='', n=1000, temperature=0.8, top_k=30):
    idx=torch.tensor([encode(prompt) or [0]],dtype=torch.long,device=device)
    for _ in range(n):
        logits,_=model(idx[:,-block_size:])
        logits=logits[:,-1,:]/temperature
        v,_=torch.topk(logits,min(top_k,logits.size(-1)))
        logits[logits<v[:,[-1]]]=-float('inf')
        probs=F.softmax(logits,-1)
        idx=torch.cat((idx,torch.multinomial(probs,1)),1)
    return decode(idx[0].tolist())

out=sample(n=1000,temperature=0.8,top_k=30)   # prompt='그는' 처럼 시작어 줄 수도 있음
open('samples.txt','w',encoding='utf-8').write(out); print(out)

## 6. 학습 (T4 GPU 기준 약 10~20분)

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)
history={'iter':[],'train':[],'val':[]}; best_val=1e9; best_iter=0
for it in range(max_iters+1):
    if it%eval_interval==0 or it==max_iters:
        L=estimate_loss(); history['iter'].append(it); history['train'].append(L['train']); history['val'].append(L['val'])
        if L['val']<best_val:                                   # val이 최저를 갱신할 때만
            best_val=L['val']; best_iter=it
            torch.save({'model':model.state_dict(),'stoi':stoi,'itos':itos,'iter':it,'val_loss':best_val},'ckpt_best.pt')  # ← best checkpoint 저장
        print(f"step {it:5d}: train {L['train']:.4f} | val {L['val']:.4f}")
    xb,yb=get_batch('train'); _,loss=model(xb,yb)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
print(f'학습 완료 — 최저 val loss {best_val:.4f} @ step {best_iter} (ckpt_best.pt 저장됨)')

## 7. 손실 곡선

In [ ]:
import matplotlib.pyplot as plt, math
plt.figure(figsize=(7,4.5))
plt.plot(history['iter'],history['train'],marker='o',ms=3,label='train')
plt.plot(history['iter'],history['val'],marker='o',ms=3,label='val')
plt.axhline(math.log(vocab_size),ls='--',c='gray',lw=1,label=f'random ln({vocab_size})={math.log(vocab_size):.2f}')
plt.xlabel('iteration'); plt.ylabel('cross-entropy loss'); plt.title('GPT on Korean literature')
plt.legend(); plt.grid(alpha=.3); plt.savefig('loss_curve.png',dpi=120); plt.show()

## 8. 텍스트 생성

In [ ]:
# 과적합된 최종 모델 대신, val 최저였던 best 모델로 생성
model.load_state_dict(torch.load('ckpt_best.pt')['model']); model.eval()
context=torch.zeros((1,1),dtype=torch.long,device=device)
out=decode(model.generate(context,2000)[0].tolist())
open('samples.txt','w',encoding='utf-8').write(out); print(out)

## 9. 결과물 저장 / 다운로드

In [ ]:
torch.save({'model':model.state_dict(),'stoi':stoi,'itos':itos,'history':history},'ckpt_last.pt')
print('저장: ckpt_best.pt(최적), ckpt_last.pt(최종), loss_curve.png, samples.txt')
try:
    from google.colab import files
    for f in ['ckpt_best.pt','loss_curve.png','samples.txt']: files.download(f)
except Exception: print('로컬이면 파일 탐색기에서 직접 받으세요.')